In [2]:

''' conda create -n paraview_env \
    -c conda-forge \
    python=3.9 \
    paraview \
    numpy scipy pandas pip 
    conda activate paraview_env 
    pvpython -m pip install ipykernel '''

# Función para verificar si ParaView está disponible
def check_paraview_python():
    """
    Verifica si ParaView Python está disponible
    """
    try:
        import paraview.simple as pv
        print("✅ ParaView Python está disponible")
        return True
    except ImportError:
        print("❌ ParaView Python no está disponible")
        return False

check_paraview_python()


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

✅ ParaView Python está disponible


In [3]:
def Create_view_white(image_size=(2048, 2048)):
    try:
        renderView = pv.GetActiveViewOrCreate('RenderView')
        renderView.ViewSize = list(image_size)
        renderView.Background = [0,0,0]  # Fondo negro
        renderView.BackgroundColorMode = 'Single Color'  
        renderView.OrientationAxesVisibility = 0  # Sin ejes
        renderView.CameraParallelProjection = 1
        ##pv.LoadPalette(paletteName='WhiteBackground')
        return renderView

    except Exception as e:
        print(f" Error en BackgroundWhite: {e}")
def calculate_perfect_parallel_scale(width, height, image_size, margin_factor=0):
    """Calcula escala paralela perfecta para llenar la imagen"""
    geom_ratio = width / height
    image_ratio = image_size[0] / image_size[1]
    
    if geom_ratio > image_ratio:
        # Geometría más ancha que imagen → width limita
        scale = (width / 2) * (1 + margin_factor)
        limiting_dimension = "WIDTH"
    else:
        # Geometría más alta que imagen → height limita  
        scale = (height / 2) * (1 + margin_factor)
        limiting_dimension = "HEIGHT"
    
    return scale  
def Camara_position(reader, renderView,image_size=(2048, 2048)):
    try:
        
        # Obtener bounds sin mostrar
        data_info = reader.GetDataInformation()
        bounds = data_info.GetBounds()

    
        width = bounds[1] - bounds[0]
        height = bounds[3] - bounds[2]
        center_x = (bounds[0] + bounds[1]) / 2
        center_y = (bounds[2] + bounds[3]) / 2
        center_z = (bounds[4] + bounds[5]) / 2
        
        # Configurar cámara perfecta
        camera = renderView.GetActiveCamera()
        camera.SetPosition([center_x, center_y, center_z + max(width, height) * 1.5])
        camera.SetFocalPoint([center_x, center_y, center_z])
        camera.SetViewUp([0, 1, 0])
        
        

        # ¡CLAVE! Escala paralela perfecta
        optimal_scale = calculate_perfect_parallel_scale(width, height, image_size, margin_factor=0.0)
        camera.SetParallelScale(optimal_scale)
        
        return camera

    except Exception as e:
        print(f"❌ Error en Camara_position: {e}")
        return None
    
############
import paraview.simple as pv
import glob
import os

def Export_folders_from_results(PATH):
    """Detect all subfolders inside PATH (simulation results)."""
    simulation_dirs = sorted([
        os.path.join(PATH, d)
        for d in os.listdir(PATH)
        if os.path.isdir(os.path.join(PATH, d))
    ])
    for s in simulation_dirs:
        print(s)
    return simulation_dirs  # opcional, útil si quieres usarlas después

#Parallel
#pip install joblib ## install the library
from joblib import Parallel, delayed
def export_one(path,variable_input='concentration'):
    print(f"🚀 Exporting PNGs from: {path}")
    export_png_from_vtu(path, variable=variable_input, min_range=0.3, max_range=0.9)
    print(f"✅ Done: {path}")
    
        
def export_png_from_vtu(vtu_file, variable = 'concentration',min_range=0, max_range=1.0):
    pv.ResetSession()
    renderView = Create_view_white()

   #look for the pvtu files
    if os.path.isdir(vtu_file):
        files = sorted(glob.glob(os.path.join(vtu_file, "Results_*.pvtu")))
    else:
        files = sorted(glob.glob(vtu_file))
        
    # Leer archivo VTU
    reader = pv.XMLPartitionedUnstructuredGridReader(FileName=files)
    reader.PointArrayStatus = [variable]
    #reader.UpdatePipeline()
    reader.TimeArray = "None"

    display = pv.Show(reader, renderView)
    pv.ColorBy(display, ('POINTS', variable))
    display.Representation = 'Surface'

    lut = pv.GetColorTransferFunction(variable, display)
    lut.ApplyPreset('Inferno (matplotlib)', True)
    lut.InvertTransferFunction()
    lut.RescaleTransferFunction(min_range, max_range)
    
    camara = Camara_position(reader,renderView) ## center the image
    pv.Render()

    #save pngs:
    out_directory = os.path.join(vtu_file, "png")
    os.makedirs(out_directory, exist_ok=True)

    """     #Aditional for contours of level set function in 0 -------------------------
    #look for the pvtu files
    if os.path.isdir(vtu_file):
        files = sorted(glob.glob(os.path.join(vtu_file, "LSF_t*.pvtu")))
    else:
        files = sorted(glob.glob(vtu_file))
        
    # Leer archivo VTU
    lsf = pv.XMLPartitionedUnstructuredGridReader(FileName=files)
    lsf.PointArrayStatus = ["ca_cyt"]
    lsf.TimeArray = "None"
    
    contour = pv.Contour(Input=lsf)
    contour.ContourBy = ['POINTS', 'ca_cyt']
    contour.Isosurfaces = [0.0]
    display_lsf = pv.Show(contour, renderView)
    pv.ColorBy(display_lsf, None)
    display_lsf.Representation = 'Surface'
    display_lsf.DiffuseColor = [0, 0, 0]
    display_lsf.LineWidth = 2
    pv.Hide(display_lsf, renderView) """

    # Guardar PNGs
    for idx, t in enumerate(reader.TimestepValues):
        print (f" Saving timestep {idx} / {len(reader.TimestepValues)}")
        renderView.ViewTime = t
        pv.Render()
        pv.SaveScreenshot(os.path.join(out_directory, f"{idx:03d}.png"), 
                        renderView,
                        #quality=100,
                        ImageResolution=renderView.ViewSize, 
                        TransparentBackground=1)

#pip install ipywidgets ## install the library
import ipywidgets as widgets
import os, glob
def create_quad_viewer(png_directories, simulation_names=None):
    """
    Flexible viewer: shows any number of simulations (3, 4, 5, ...)
    in a grid layout with individual and synchronized controls.
    """

    all_png_files = []
    all_widgets = []
    all_sliders = []
    all_labels = []
    
    # Load images for each simulation
    for i, png_dir in enumerate(png_directories):
        png_files = sorted(glob.glob(os.path.join(png_dir, "*.png")))
        print(f"Found {len(png_files)} PNG files in {png_dir}")
        all_png_files.append(png_files)
        
        image_widget = widgets.Image(
            value=open(png_files[0], "rb").read(),
            format='png',
            width=400,
            height=300
        )
        all_widgets.append(image_widget)
        
        slider = widgets.IntSlider(
            value=0,
            min=0,
            max=len(png_files)-1,
            step=1,
            description=f'{simulation_names[i]}:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='400px')
        )
        all_sliders.append(slider)
        
        label = widgets.Label(
            value=f"Timestep 0/{len(png_files)-1}: {os.path.basename(png_files[0])}",
            layout=widgets.Layout(width='400px'),   
        )
        all_labels.append(label)
        
        # Update function (closure)
        def make_update_function(idx):
            def update_image(change):
                frame_idx = change['new']
                files = all_png_files[idx]
                with open(files[frame_idx], "rb") as f:
                    all_widgets[idx].value = f.read()
                all_labels[idx].value = f"Timestep {frame_idx}/{len(files)-1}: {os.path.basename(files[frame_idx])}"
            return update_image
        slider.observe(make_update_function(i), names='value')
    
    # Master control slider
    master_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=min([len(files) for files in all_png_files])-1,
        step=1,
        description='Sync All:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='100%')
    )

    def sync_all(change):
        frame_idx = change['new']
        for slider in all_sliders:
            if frame_idx <= slider.max:
                slider.value = frame_idx
    master_slider.observe(sync_all, names='value')
    
    rows = []
    for i in range(0, len(all_widgets), 2):
        imgs = all_widgets[i:i+2]
        lbls = all_labels[i:i+2]
        slds = all_sliders[i:i+2]
        rows.append(widgets.HBox(lbls,layout=widgets.Layout(justify_content='center'),    width='100%',height='auto'))
        rows.append(widgets.HBox(imgs,layout=widgets.Layout(justify_content='center'),   width='100%', height='auto'))
        rows.append(widgets.HBox(slds,layout=widgets.Layout(justify_content='center'),   width='100%', height='auto'))

    viewer = widgets.VBox([
        widgets.HTML("<h3>Timesteps</h3>"),
        master_slider,
        widgets.HTML("<h3>Simulations</h3>"),
        *rows
    ])
    
    return viewer
 
 
 
from PIL import Image
import matplotlib.pyplot as plt
def plot_timesteps_images(folder, start, end, step=10, ncols=3, figsize=(12, 6), pad=3):
    """
    Displays a grid of PNG images for selected timesteps (with zero-padded filenames).

    Parameters
    ----------
    folder : str
        Directory where images are stored (named like 000.png, 001.png, ...).
    start : int
        First timestep to display.
    end : int
        Last timestep to display.
    step : int, optional
        Step interval between images (default = 10).
    ncols : int, optional
        Number of columns in the plot grid.
    figsize : tuple, optional
        Figure size for matplotlib.
    pad : int, optional
        Number of zero-padding digits (e.g., 3 → 000, 001, 020).
    """
    timesteps = list(range(start, end + 1, step))
    nrows = (len(timesteps) + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()
    
    for i, t in enumerate(timesteps):
        filename = os.path.join(folder, f"{t:0{pad}d}.png")
        if os.path.exists(filename):
            img = Image.open(filename)
            axes[i].imshow(img)
            axes[i].set_title(f"Timestep {t}")
            axes[i].axis("off")
        else:
            axes[i].text(0.5, 0.5, f"Missing\n{t:0{pad}d}.png", ha='center', va='center')
            axes[i].axis("off")
    
    # Hide extra axes
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    
    plt.tight_layout()
    plt.show()

In [10]:
PATH = "Test1"  # Folder where all sims are located
Directorio_results = Export_folders_from_results(PATH)
## add "/Simulation2D" in the Directorio_results list for each simulation
Directorio_results = [d + "/Simulation2D" for d in Directorio_results]
Directorio_results = Directorio_results[1:] 


Test1/TimingResults
Test1/procesors192
Test1/procesors384
Test1/procesors450
Test1/procesors48
Test1/procesors576
Test1/procesors96
a


In [14]:
Parallel(n_jobs=4)(delayed(export_one)(p, variable_input='t') for p in Directorio_results)


🚀 Exporting PNGs from: Test1/procesors384/Simulation2D
 Saving timestep 0 / 5
 Saving timestep 1 / 5
 Saving timestep 2 / 5
 Saving timestep 3 / 5
 Saving timestep 4 / 5
✅ Done: Test1/procesors384/Simulation2D
🚀 Exporting PNGs from: Test1/procesors576/Simulation2D
 Saving timestep 0 / 8
 Saving timestep 1 / 8
 Saving timestep 2 / 8
 Saving timestep 3 / 8
 Saving timestep 4 / 8
 Saving timestep 5 / 8
 Saving timestep 6 / 8
 Saving timestep 7 / 8
✅ Done: Test1/procesors576/Simulation2D
🚀 Exporting PNGs from: Test1/procesors192/Simulation2D
 Saving timestep 0 / 25
 Saving timestep 1 / 25
 Saving timestep 2 / 25
 Saving timestep 3 / 25
 Saving timestep 4 / 25
 Saving timestep 5 / 25
 Saving timestep 6 / 25
 Saving timestep 7 / 25
 Saving timestep 8 / 25
 Saving timestep 9 / 25
 Saving timestep 10 / 25
 Saving timestep 11 / 25
 Saving timestep 12 / 25
 Saving timestep 13 / 25
 Saving timestep 14 / 25
 Saving timestep 15 / 25
 Saving timestep 16 / 25
 Saving timestep 17 / 25
 Saving timestep

/scratch/flore0a/iops/miniconda3/envs/pv_env/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


🚀 Exporting PNGs from: Test1/procesors450/Simulation2D
 Saving timestep 0 / 11
 Saving timestep 1 / 11
 Saving timestep 2 / 11
 Saving timestep 3 / 11
 Saving timestep 4 / 11
 Saving timestep 5 / 11
 Saving timestep 6 / 11
 Saving timestep 7 / 11
 Saving timestep 8 / 11
 Saving timestep 9 / 11
 Saving timestep 10 / 11
✅ Done: Test1/procesors450/Simulation2D
🚀 Exporting PNGs from: Test1/procesors96/Simulation2D
 Saving timestep 0 / 13
 Saving timestep 1 / 13
 Saving timestep 2 / 13
 Saving timestep 3 / 13
 Saving timestep 4 / 13
 Saving timestep 5 / 13
 Saving timestep 6 / 13
 Saving timestep 7 / 13
 Saving timestep 8 / 13
 Saving timestep 9 / 13
 Saving timestep 10 / 13
 Saving timestep 11 / 13
 Saving timestep 12 / 13
✅ Done: Test1/procesors96/Simulation2D


[None, None, None, None, None, None]

🚀 Exporting PNGs from: Test1/procesors450/Simulation2D
 Saving timestep 0 / 11
 Saving timestep 1 / 11
 Saving timestep 2 / 11
 Saving timestep 3 / 11
 Saving timestep 4 / 11
 Saving timestep 5 / 11
 Saving timestep 6 / 11
 Saving timestep 7 / 11
 Saving timestep 8 / 11
 Saving timestep 9 / 11
 Saving timestep 10 / 11
✅ Done: Test1/procesors450/Simulation2D
🚀 Exporting PNGs from: Test1/procesors192/Simulation2D
 Saving timestep 0 / 25
 Saving timestep 1 / 25
 Saving timestep 2 / 25
 Saving timestep 3 / 25
 Saving timestep 4 / 25
 Saving timestep 5 / 25
 Saving timestep 6 / 25
 Saving timestep 7 / 25
 Saving timestep 8 / 25
 Saving timestep 9 / 25
 Saving timestep 10 / 25
 Saving timestep 11 / 25
 Saving timestep 12 / 25
 Saving timestep 13 / 25
 Saving timestep 14 / 25
 Saving timestep 15 / 25
 Saving timestep 16 / 25
 Saving timestep 17 / 25
 Saving timestep 18 / 25
 Saving timestep 19 / 25
 Saving timestep 20 / 25
 Saving timestep 21 / 25
 Saving timestep 22 / 25
 Saving timestep 23 

In [15]:
#how to remove the previos before /
print(Directorio_results)
#remove Test1/TimingResults/Simulation2D (first in the list)from Directorio_results
Directorio_results = Directorio_results[1:] 


name = [ os.path.basename(os.path.dirname(path)) for path in Directorio_results]
png_directory = [d + "/png" for d in Directorio_results] # add the /png folder
print(png_directory)

quad_viewer = create_quad_viewer(png_directory, name)
display(quad_viewer)

['Test1/procesors192/Simulation2D', 'Test1/procesors384/Simulation2D', 'Test1/procesors450/Simulation2D', 'Test1/procesors48/Simulation2D', 'Test1/procesors576/Simulation2D', 'Test1/procesors96/Simulation2D']
['Test1/procesors384/Simulation2D/png', 'Test1/procesors450/Simulation2D/png', 'Test1/procesors48/Simulation2D/png', 'Test1/procesors576/Simulation2D/png', 'Test1/procesors96/Simulation2D/png']
Found 5 PNG files in Test1/procesors384/Simulation2D/png
Found 11 PNG files in Test1/procesors450/Simulation2D/png
Found 14 PNG files in Test1/procesors48/Simulation2D/png
Found 8 PNG files in Test1/procesors576/Simulation2D/png
Found 13 PNG files in Test1/procesors96/Simulation2D/png
